In [17]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional, Set

import numpy as np
import pandas as pd
from pymongo import MongoClient
import yaml


In [18]:
def load_config(path: str | Path = "../config/config.yaml") -> dict[str, Any]:
    path = Path(path)
    if not path.exists():
        path = Path("config/config.yaml")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


config = load_config()
mongo_config = config["mongo"]

In [19]:
client = MongoClient(mongo_config.get('url'))
db = client[mongo_config.get('db')]
col_schedule = db[mongo_config.get('collection').get('collection_schedule')]
season = '2009-2010'
df_schedule = pd.DataFrame(col_schedule.find({'season': season}))
check_games = df_schedule['game_id'].sample(5, random_state=1909).values.tolist()

In [20]:
col_team_stats = db[mongo_config.get('collection').get('collection_team_game_stats')]
col_player_stats = db[mongo_config.get('collection').get('collection_player_game_stats')]

df_team_stats = pd.DataFrame(col_team_stats.find({'game_id': {"$in": check_games}}))
df_player_stast = pd.DataFrame(col_player_stats.find({'game_id': {"$in": check_games}}))

In [21]:
pd.set_option("display.max_rows", None)

for game in check_games:
    team_stats = df_team_stats[df_team_stats.game_id == game]
    team_stats = team_stats.sort_values(by='game_venue', ascending=False).reset_index(drop=True)
    game_info = team_stats[['game_id', 'game_date', 'week', 'team_name', 'game_venue']]
    stats = pd.json_normalize(team_stats.stats)
    stats_df = stats[[col for col in stats.columns if 'ft.' in col]]
    
    final_game = pd.merge(
        game_info,
        stats_df,
        left_index=True,
        right_index=True
    )
    display(final_game.T)
    print("\n" + "-" * 80 + "\n")


,0,1
game_id,328154,328154
game_date,2009-12-19 14:30:00,2009-12-19 14:30:00
week,17,17
team_name,Eintracht Frankfurt,Wolfsburg
game_venue,home,away
ft.possession,0.517602,0.482398
ft.goals,2,2
ft.shots,12,13
ft.shot_goal,2,2
ft.shot_on_target,4,5



--------------------------------------------------------------------------------



,0,1
game_id,328236,328236
game_date,2010-03-13 14:30:00,2010-03-13 14:30:00
week,26,26
team_name,Bochum,Borussia Dortmund
game_venue,home,away
ft.possession,0.392665,0.607335
ft.goals,1,4
ft.shots,11,13
ft.shot_goal,1,4
ft.shot_on_target,2,9



--------------------------------------------------------------------------------



,0,1
game_id,328189,328189
game_date,2010-02-06 14:30:00,2010-02-06 14:30:00
week,21,21
team_name,FC Koln,Hamburger SV
game_venue,home,away
ft.possession,0.465863,0.534137
ft.goals,3,3
ft.shots,17,14
ft.shot_goal,3,3
ft.shot_on_target,8,9



--------------------------------------------------------------------------------



,0,1
game_id,328287,328287
game_date,2010-04-18 16:30:00,2010-04-18 16:30:00
week,31,31
team_name,Eintracht Frankfurt,Hertha Berlin
game_venue,home,away
ft.possession,0.492872,0.507128
ft.goals,2,2
ft.shots,10,10
ft.shot_goal,2,2
ft.shot_on_target,7,7



--------------------------------------------------------------------------------



,0,1
game_id,328251,328251
game_date,2010-03-27 14:30:00,2010-03-27 14:30:00
week,28,28
team_name,Bayern Munich,VfB Stuttgart
game_venue,home,away
ft.possession,0.569143,0.430857
ft.goals,1,2
ft.shots,9,15
ft.shot_goal,1,2
ft.shot_on_target,3,8



--------------------------------------------------------------------------------



In [22]:
mongo_config

{'backup_folder': './mongo_backup',
 'db': 'WhoScored',
 'url': 'mongodb://localhost:27017/',
 'collection': {'collection_teams': 'available_teams',
  'collection_schedule': 'game_schedule',
  'collection_logs': 'error_logs',
  'collection_raw_events': 'game_raw_events',
  'collection_processed_events': 'game_processed_events',
  'collection_team_game_stats': 'game_team_stats',
  'collection_player_game_stats': 'game_player_stats'}}

In [ ]:
# db[mongo_config.get('collection').get("collection_schedule")].delete_many({})
# db[mongo_config.get('collection').get("collection_raw_events")].delete_many({})
# db[mongo_config.get('collection').get("collection_processed_events")].delete_many({})
# db[mongo_config.get('collection').get("collection_team_game_stats")].delete_many({})
# db[mongo_config.get('collection').get("collection_player_game_stats")].delete_many({})

In [29]:
game = pd.DataFrame(db[mongo_config.get('collection').get("collection_raw_events")].find({"game_id": 328154}))

In [34]:
game[game.is_goal == True]

,_id,game_id,away_team_id,away_team_name,competition_country,competition_name,game_date,game_status,home_team_id,home_team_name,...,blocked_x,blocked_y,qualifiers,is_touch,is_shot,is_goal,card_type,related_event_id,related_player_id,event_idx
502,6a09bef5f73bd6958dd7d45f,328154,33,Wolfsburg,Germany,Bundesliga,2009-12-19 14:30:00,finished,45,Eintracht Frankfurt,...,NaN,NaN,"[{'type': {'displayName': 'BoxCentre', 'value'...",True,True,True,NaN,293.0,6268.0,502
671,6a09bef5f73bd6958dd7d508,328154,33,Wolfsburg,Germany,Bundesliga,2009-12-19 14:30:00,finished,45,Eintracht Frankfurt,...,NaN,NaN,"[{'type': {'displayName': 'GoalMouthY', 'value...",True,True,True,NaN,346.0,11269.0,671
1233,6a09bef5f73bd6958dd7d73a,328154,33,Wolfsburg,Germany,Bundesliga,2009-12-19 14:30:00,finished,45,Eintracht Frankfurt,...,NaN,NaN,"[{'type': {'displayName': 'GoalMouthY', 'value...",True,True,True,NaN,NaN,NaN,1233
1383,6a09bef5f73bd6958dd7d7d0,328154,33,Wolfsburg,Germany,Bundesliga,2009-12-19 14:30:00,finished,45,Eintracht Frankfurt,...,NaN,NaN,"[{'type': {'displayName': 'OutOfBoxCentre', 'v...",True,True,True,NaN,751.0,12392.0,1383


In [35]:
game[game.is_goal == True]['qualifiers'].values

array([list([{'type': {'displayName': 'BoxCentre', 'value': 17}}, {'type': {'displayName': 'Head', 'value': 15}}, {'type': {'displayName': 'Assisted', 'value': 29}}, {'type': {'displayName': 'GoalMouthZ', 'value': 103}, 'value': '12.5'}, {'type': {'displayName': 'Zone', 'value': 56}, 'value': 'Center'}, {'type': {'displayName': 'LowRight', 'value': 80}}, {'type': {'displayName': 'RelatedEventId', 'value': 55}, 'value': '293'}, {'type': {'displayName': 'SetPiece', 'value': 24}}, {'type': {'displayName': 'GoalMouthY', 'value': 102}, 'value': '46.7'}, {'type': {'displayName': 'IntentionalAssist', 'value': 154}}]),
       list([{'type': {'displayName': 'GoalMouthY', 'value': 102}, 'value': '50.9'}, {'type': {'displayName': 'GoalMouthZ', 'value': 103}, 'value': '2.8'}, {'type': {'displayName': 'Assisted', 'value': 29}}, {'type': {'displayName': 'BoxCentre', 'value': 17}}, {'type': {'displayName': 'Zone', 'value': 56}, 'value': 'Center'}, {'type': {'displayName': 'IntentionalAssist', 'value'